# 02 — Prepare Hyperscaler Candidates

## Research purpose

Clean the OSM records and identify candidates associated with Amazon, Google and Microsoft using the manual classification rules.


## Imports

Load the libraries used for table processing, tag parsing and operator classification.


In [1]:
import json
import math
from pathlib import Path
import re

import pandas as pd

## Project location and paths

Identify the project directory and define the paths for the normalised OSM records, manual rules and candidate tables.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_MANUAL = PROJECT_ROOT / "data" / "manual"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

raw_path = DATA_INTERIM / "osm_datacenters_normalized.csv"

## Operator evidence settings

Define which OSM tags can provide operator evidence and which contain source or import metadata. This keeps metadata mentions from being used to identify a facility operator.


In [3]:
EVIDENCE_KEYS = {"operator", "owner", "brand", "name", "official_name", "short_name", "alt_name", "website", "contact:website", "url", "tenant", "description"}
EVIDENCE_PREFIXES = ("operator:", "owner:", "brand:", "tenant:")
METADATA_KEYS = {"source", "source_ref", "attribution", "created_by", "import", "import_uuid"}
METADATA_PREFIXES = ("source:", "import:", "tiger:", "gnis:")

## Load OSM records

Load the normalised OSM table and check that it contains the source identifiers, coordinates and tags needed for classification.


In [4]:
df = pd.read_csv(raw_path)
required = {"source_id", "osm_ref", "osm_id", "osm_type", "latitude", "longitude", "osm_tags"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Normalized OSM input is missing {missing}")
print(f"Normalized OSM records: {len(df):,}")

Normalized OSM records: 4,286


## Clean attributes and coordinates

Convert coordinates and polygon areas to numeric values and extract the facility attributes from the OSM tags. Separate records with missing or out-of-range coordinates and save both the clean and rejected tables.


In [5]:
def parse_tags(value):
    if pd.isna(value) or value == "":
        return {}
    try:
        return json.loads(value)
    except Exception:
        return {}

for column in ["latitude", "longitude", "polygon_area_m2"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

df["tags_dict"] = df["osm_tags"].apply(parse_tags)
df["facility_name_raw"] = df["name"].fillna("")
df["operator_raw"] = df["operator"].fillna("")
df["owner_raw"] = df["owner"].fillna("")
df["brand_raw"] = df["brand"].fillna("")
df["website_raw"] = df["website"].fillna("")
df["building_tag"] = df["tags_dict"].apply(lambda tags: tags.get("building"))
df["landuse_tag"] = df["tags_dict"].apply(lambda tags: tags.get("landuse"))
df["levels_raw"] = df["tags_dict"].apply(lambda tags: tags.get("building:levels"))
df["height_raw"] = df["tags_dict"].apply(lambda tags: tags.get("height"))
df["source_type"] = "osm"

invalid_coordinates = df["latitude"].isna() | df["longitude"].isna() | ~df["latitude"].between(-90, 90) | ~df["longitude"].between(-180, 180)
rejected = df[invalid_coordinates].copy()
rejected["rejection_reason"] = "missing_or_invalid_coordinates"
clean = df[~invalid_coordinates].copy()

clean = clean.drop(columns=["tags_dict"])
rejected = rejected.drop(columns=["tags_dict"])

clean.to_csv(DATA_INTERIM / "datacenters_osm_clean.csv", index=False)
rejected.to_csv(DATA_INTERIM / "datacenters_osm_rejected.csv", index=False)

print(f"Clean rows: {len(clean):,}")
print(f"Rejected rows: {len(rejected):,}")
clean[["source_id", "osm_type", "facility_name_raw", "operator_raw", "geometry_kind", "polygon_area_m2"]].head()

Clean rows: 4,286
Rejected rows: 0


,source_id,osm_type,facility_name_raw,operator_raw,geometry_kind,polygon_area_m2
0,osm_node_599587508,node,Cellnex,,point,NaN
1,osm_node_688791288,node,Telstra,Telstra,point,NaN
2,osm_node_717740190,node,Equinix AM7,Equinix,point,NaN
3,osm_node_751881301,node,CoreSite DC1,CoreSite,point,NaN
4,osm_node_1530992851,node,,ATW,point,NaN


## Load classification rules

Load the manual operator rules, keep the active entries and sort them by priority.


In [6]:
rules = pd.read_csv(DATA_MANUAL / "operator_classification_rules.csv").sort_values("priority")
rules["active"] = rules["active"].astype(str).str.lower().eq("true")
rules = rules[rules["active"]].copy()

## Classify operator evidence

Define functions to extract permitted operator evidence and apply the classification rules in priority order. The first hyperscaler match determines the operator, while unmatched records remain outside the sample. Retain multiple matches and metadata-only mentions for the classification audit.


In [7]:
# Recognize missing scalar evidence before combining source fields.
def is_missing(value):
    return value is None or (isinstance(value, float) and math.isnan(value))

# Extract permitted operator evidence or source-metadata values from JSON tags.
def tag_values(value, evidence=True):
    if is_missing(value) or not str(value).strip().startswith("{"):
        return None
    try:
        tags = json.loads(str(value))
    except json.JSONDecodeError:
        return None
    values = []
    for key, item in tags.items():
        normalized = str(key).strip().lower()
        is_metadata = normalized in METADATA_KEYS or normalized.startswith(METADATA_PREFIXES)
        keep = (normalized in EVIDENCE_KEYS or normalized.startswith(EVIDENCE_PREFIXES)) and not is_metadata
        if (keep if evidence else is_metadata) and item is not None and str(item).strip():
            values.append(f"{key}={item}")
    return values

# Combine nonempty operator evidence from the supplied record fields.
def evidence_values(*values):
    output = []
    for value in values:
        if is_missing(value):
            continue
        parsed = tag_values(value, evidence=True)
        output.extend(parsed if parsed is not None else [str(value).strip()])
    return [value for value in output if value]

# Record hyperscaler mentions confined to source/import metadata in the audit.
def metadata_mentions_hyperscaler(value):
    metadata = tag_values(value, evidence=False) or []
    text = " | ".join(metadata).lower()
    patterns = rules.loc[rules["rule_type"].eq("hyperscaler"), "pattern"]
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)

# Apply ordered rules and retain the selected operator, confidence, and match evidence.
def classify_operator(*values):
    evidence = evidence_values(*values)
    text = " | ".join(evidence).lower()
    matched = rules[rules["pattern"].map(lambda pattern: bool(re.search(pattern, text, flags=re.IGNORECASE)))]
    hyperscaler = matched[matched["rule_type"].eq("hyperscaler")]
    groups = sorted(hyperscaler["operator_group"].dropna().unique())
    if not hyperscaler.empty:
        first = hyperscaler.iloc[0]
        return {
            "operator_standardized": first["operator_standardized"],
            "operator_group": first["operator_group"],
            "is_hyperscaler": True,
            "hyperscaler_category": first["hyperscaler_category"],
            "operator_match_method": f"regex:{first['pattern']}",
            "operator_match_confidence": first["match_confidence"],
            "operator_match_evidence": " | ".join(evidence),
            "operator_match_count": len(hyperscaler),
            "operator_match_groups": "|".join(groups),
        }
    return {
        "operator_standardized": None,
        "operator_group": None,
        "is_hyperscaler": False,
        "hyperscaler_category": "not_in_scope",
        "operator_match_method": "no_match",
        "operator_match_confidence": "unknown",
        "operator_match_evidence": " | ".join(evidence),
        "operator_match_count": 0,
        "operator_match_groups": "",
    }

## Save candidates and classification audit

Apply the operator rules to all input records and select hyperscaler matches with valid coordinates. Save the candidates with their source identifiers and a separate audit containing the original tags and classification evidence. Display the operator and audit counts to inspect the selected sample.


In [8]:
matches = df.apply(
    lambda row: classify_operator(
        row.get("operator_raw"),
        row.get("owner_raw"),
        row.get("brand_raw"),
        row.get("facility_name_raw"),
        row.get("osm_tags"),
    ),
    axis=1,
)

match_df = pd.DataFrame(matches.tolist())
df = pd.concat([df.reset_index(drop=True), match_df], axis=1)
df["operator_metadata_mentions_hyperscaler"] = df["osm_tags"].apply(metadata_mentions_hyperscaler)

# Operator matches take precedence; check metadata only for the remaining rows.
selected_hyperscaler = df["is_hyperscaler"].map(bool)
metadata_only = df["operator_metadata_mentions_hyperscaler"].where(~selected_hyperscaler, False).map(bool)
df["operator_audit_flag"] = "not_in_scope"
df.loc[metadata_only, "operator_audit_flag"] = "metadata_mentions_hyperscaler_only"
df.loc[selected_hyperscaler, "operator_audit_flag"] = "selected_hyperscaler"

# Classification keeps the row order but resets the index; align the coordinate mask.
valid_coordinates = ~invalid_coordinates.reset_index(drop=True)
candidates = df[(df["is_hyperscaler"] == True) & valid_coordinates].copy()

audit_columns = [
    "source_id",
    "osm_ref",
    "osm_type",
    "facility_name_raw",
    "operator_raw",
    "owner_raw",
    "brand_raw",
    "operator_standardized",
    "operator_group",
    "is_hyperscaler",
    "hyperscaler_category",
    "operator_match_method",
    "operator_match_confidence",
    "operator_match_evidence",
    "operator_match_count",
    "operator_match_groups",
    "operator_metadata_mentions_hyperscaler",
    "operator_audit_flag",
    "osm_tags",
]
operator_audit = df[audit_columns].copy()

# Keep classification evidence in the audit rather than carrying it into facility estimation.
candidates = candidates.drop(columns=[
    "operator_raw", "is_hyperscaler", "hyperscaler_category", "osm_tags", "tags_dict",
])

candidates.to_csv(DATA_INTERIM / "datacenters_hyperscaler_candidates.csv", index=False)
operator_audit.to_csv(DATA_INTERIM / "datacenters_operator_audit.csv", index=False)

print(candidates["operator_group"].value_counts(dropna=False))
print(df["operator_audit_flag"].value_counts(dropna=False))
candidates[["source_id", "facility_name_raw", "operator_group", "operator_match_confidence", "operator_match_evidence"]].head()

operator_group
Amazon       308
Microsoft    113
Google        90
Name: count, dtype: int64
operator_audit_flag
not_in_scope                          3749
selected_hyperscaler                   511
metadata_mentions_hyperscaler_only      26
Name: count, dtype: int64


,source_id,facility_name_raw,operator_group,operator_match_confidence,operator_match_evidence
80,osm_node_4879481820,Google Fiber,Google,high,Google | Google Fiber | name=Google Fiber | op...
267,osm_node_8208591051,Amazon SIN4,Amazon,high,Amazon SIN4 | name=Amazon SIN4
268,osm_node_8208591054,Amazon SIN2,Amazon,high,Amazon SIN2 | name=Amazon SIN2
269,osm_node_8208591112,Google Singapore DC,Google,high,Google Singapore DC | name=Google Singapore DC...
282,osm_node_8208598531,Amazon SIN3,Amazon,high,Amazon SIN3 | name=Amazon SIN3
